In [14]:
import pandas as pd
import requests
import holidays


# WEATHER FUNCTION

def get_weather(city, lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2025-01-01",
        "end_date": "2025-12-31",
        "daily": [
            "temperature_2m_mean",
            "precipitation_sum",
            "relative_humidity_2m_mean",
            "windspeed_10m_max"
        ],
        "timezone": "Europe/Sofia"
    }

    r = requests.get(url, params=params)
    data = r.json()["daily"]

    df = pd.DataFrame(data)
    df["city"] = city
    return df


# CITIES (ALL)

cities = {
    "Varna": (43.2141, 27.9147),
    "Burgas": (42.5048, 27.4626),
    "Shumen": (43.2712, 26.9361),
    "Ruse": (43.8356, 25.9657),
    "Sliven": (42.6818, 26.3227),
    "Veliko Tarnovo": (43.0757, 25.6172),
    "Yambol": (42.4840, 26.5030),
    "Targovishte": (43.2512, 26.5729),
    "Razgrad": (43.5333, 26.5167),
    "Dobrich": (43.5667, 27.8333),
    "Silistra": (44.1167, 27.2667),
    "Stara Zagora": (42.4258, 25.6345),
    "Plovdiv": (42.1354, 24.7453),
    "Pleven": (43.4170, 24.6067),
    "Gabrovo": (42.8747, 25.3342)
}


# DOWNLOAD DATA

all_data = []

for city, (lat, lon) in cities.items():
    print(f"Downloading {city}...")
    df = get_weather(city, lat, lon)
    all_data.append(df)

final_df = pd.concat(all_data, ignore_index=True)


# CLEAN + RENAME WITH UNITS

final_df = final_df.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temperature_celsius",
    "precipitation_sum": "precipitation_mm",
    "relative_humidity_2m_mean": "humidity_percent",
    "windspeed_10m_max": "wind_speed_kmh"
})

final_df["date"] = pd.to_datetime(final_df["date"])


# HOLIDAYS (BG NATIONAL)

bg_holidays = holidays.BG(years=2025)
final_df["is_holiday"] = final_df["date"].dt.date.apply(lambda x: x in bg_holidays)


# WEEKEND

final_df["is_weekend"] = final_df["date"].dt.weekday >= 5


# NON-WORKING DAY

final_df["is_non_working_day"] = (
    final_df["is_weekend"] | final_df["is_holiday"]
)


# SORT

final_df = final_df.sort_values(["city", "date"]).reset_index(drop=True)


# SAVE

output_path = "data/weather_dataset_bg.csv"
final_df.to_csv(output_path, index=False, sep=",")

print("DONE!")
print(f"Saved to: {output_path}")
print(final_df.head())

DONE!
Saved to: data/weather_dataset_bg.csv
        date  temperature_celsius  precipitation_mm  humidity_percent  \
0 2025-01-01                  2.9               0.0                86   
1 2025-01-02                  4.2               0.0                86   
2 2025-01-03                  6.5               0.0                87   
3 2025-01-04                  4.7               1.0                74   
4 2025-01-05                  1.4               0.0                84   

   wind_speed_kmh    city  is_holiday  is_weekend  is_non_working_day  
0            11.1  Burgas        True       False                True  
1            16.7  Burgas       False       False               False  
2            17.5  Burgas       False       False               False  
3            14.1  Burgas       False        True                True  
4             6.0  Burgas       False        True                True  
